In [1]:
import torch, numpy as np
print(torch.__version__)         # deve mostrar 2.1.x+cu117 (ou cu118)
print(torch.cuda.is_available()) # True
print(np.__version__)  

2.5.1
True
1.26.4


In [2]:
import sys
import types
import importlib.machinery

# cria um módulo fake "bitsandbytes" com __spec__ válido e um submódulo nn vazio
fake_bnb = types.ModuleType("bitsandbytes")
fake_bnb.__spec__ = importlib.machinery.ModuleSpec(name="bitsandbytes", loader=None)
fake_bnb.nn = types.SimpleNamespace()
sys.modules["bitsandbytes"] = fake_bnb

In [3]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
import transformers, importlib

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling
)

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper
import torch
import transformers


# def top_k_top_p_filtering(
#     logits: torch.Tensor,
#     top_k: int = 0,
#     top_p: float = 1.0,
#     filter_value: float = -float("Inf"),
#     min_tokens_to_keep: int = 1,
# ):
#     """
#     Reimplementação mínima usada pelo TRL 0.7.
#     Aplica top-k e/ou nucleus (top-p) nos logits em lote.
#     """
#     if top_k > 0:
#         logits = TopKLogitsWarper(top_k=top_k, min_tokens_to_keep=min_tokens_to_keep)(
#             None, logits
#         )
#     if 0 < top_p < 1.0:
#         logits = TopPLogitsWarper(top_p=top_p, min_tokens_to_keep=min_tokens_to_keep)(
#             None, logits
#         )
#     return logits


# # injeta no namespace que o TRL espera
# transformers.top_k_top_p_filtering = top_k_top_p_filtering

In [5]:
from trl import SFTTrainer
from seqeval.metrics import f1_score, classification_report
from tqdm.auto import tqdm

In [6]:
# para TF32 em matmuls (Ampere+)
torch.backends.cuda.matmul.allow_tf32 = True  
# para otimização de cudnn (caso shape de batch seja constante)
torch.backends.cudnn.benchmark = True

# Configuração e Verificação Inicial

In [7]:
JSON_PATH = "data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
BASE_MODEL = "gpt2-large"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# cada entrada já tem tokens + ner_tokens

In [8]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [9]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token  

# Splits

In [11]:
def holdout_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    idx = np.arange(len(dataset))
    tr, te = train_test_split(idx, test_size=pct_test, random_state=seed)
    return {"train": dataset.select(tr), "test": dataset.select(te)}


def loc_split(dataset: Dataset, pct_test: float = 0.2, ngram: int = 4, seed: int = 42):
    docs = [" ".join(t) for t in dataset["tokens"]]
    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs).astype(bool)
    inter = X @ X.T
    sz = X.sum(1).A1
    union = (sz[:, None] + sz[None, :]) - inter.A
    np.fill_diagonal(union, 1)
    jac = (inter.A / union).mean(1)  # overlap médio
    order = np.argsort(jac)  # menor → teste
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [12]:
def len_split(
    dataset: Dataset,
    pct_test: float = 0.2,
    seed: int = 42,
    longest_as_test: bool = True,
):
    """Teste = sentenças mais longas (default) ou mais curtas."""
    lengths = np.array([len(t) for t in dataset["tokens"]])
    order = np.argsort(-lengths) if longest_as_test else np.argsort(lengths)
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [13]:
def split_heur_length(ds, top_pct=0.20):
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))
    idx_long = np.where(lengths >= thr)[0]
    idx_short = np.where(lengths < thr)[0]
    return DatasetDict(
        train=ds.select(idx_short.tolist()), dev=ds.select(idx_long.tolist())
    )

In [14]:
def rarity_score(labels_seq, freq_dict):
    # raridade = soma(1/freq) dos tipos únicos na sentença
    seen = {lab[2:] for lab in labels_seq if lab != "O"}
    return sum(1 / freq_dict[t] for t in seen) if seen else 0.0


def rarity_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    # ❶ contar frequências globais por tipo
    all_types = [lab[2:] for seq in dataset["ner_tokens"] for lab in seq if lab != "O"]
    freq = Counter(all_types)
    # ❷ calcular score de cada sentença
    scores = np.array([rarity_score(seq, freq) for seq in dataset["ner_tokens"]])
    order = np.argsort(-scores)  # mais raras primeiro
    n_test = int(len(dataset) * pct_test)
    te, tr = order[:n_test], order[n_test:]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [15]:
# ---------- 1.3 Reverse-Curriculum easy→hard ------------------------
def difficulty_scores(dataset: Dataset):
    length = np.array([len(t) for t in dataset["tokens"]])
    dens = np.array(
        [sum(l != "O" for l in labs) / len(labs) for labs in dataset["ner_tokens"]]
    )
    return (length - length.mean()) / length.std() + (dens - dens.mean()) / dens.std()


def curriculum_split(dataset: Dataset, pct_test: float = 0.2, seed: int = 42):
    s = difficulty_scores(dataset)
    order = np.argsort(s)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    te, tr = order[-n_test:], order[:-n_test]
    return {"train": dataset.select(tr), "test": dataset.select(te)}

In [16]:
ACTIVE_SPLITS = {
    #"holdout": holdout_split,
    #"loc": loc_split,
    #"curriculum": curriculum_split,
    "len": len_split,  # ← heurística de tamanho
    #"rarity": rarity_split,  # ← heurística de raridade
}

# Tokenização e Métricas

In [17]:
def inline_tags(tokens, labels):
    out, open_tag = [], None
    for tok, lab in zip(tokens, labels):
        if lab.startswith("B-"):
            if open_tag:
                out.append(f"</{open_tag}>")
            open_tag = lab[2:]
            out.append(f"<{open_tag}>{tok}")
        elif lab.startswith("I-"):
            out.append(tok)
        else:
            if open_tag:
                out.append(f"</{open_tag}>")
                open_tag = None
            out.append(tok)
    if open_tag:
        out.append(f"</{open_tag}>")
    return " ".join(out)

In [18]:
def build_sft_dataset(raw_ds):
    rows = []
    for ex in raw_ds:
        sent = " ".join(ex["tokens"])
        tagged = inline_tags(ex["tokens"], ex["ner_tags"])
        prompt = "You are an NER tagger for Portuguese.\n" f"Sentence: {sent}\nTags:"
        rows.append({"text": prompt + " " + tagged})  # ← uma coluna só
    return Dataset.from_list(rows)

In [19]:
def load_base_model_fp16():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
    )
    model.gradient_checkpointing_enable()
    model.to("cuda")  # <-- lança AssertionError se torch.cuda não estiver disponível
    return model


# def load_base_model_fp16():
#     model = AutoModelForCausalLM.from_pretrained(
#         BASE_MODEL,
#         torch_dtype=torch.float16,
#     )
#     model.gradient_checkpointing_enable()
#     return model

In [20]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
from trl import SFTConfig, SFTTrainer

for split_name, split_fn in ACTIVE_SPLITS.items():
    print(f"\n=== Split: {split_name} ===")
    parts    = split_fn(geocorpus_full)
    ds_train = build_sft_dataset(parts["train"])
    ds_test  = build_sft_dataset(parts["test"])

    # 1️⃣ Load base FP16 + LoRA
    base_model = load_base_model_fp16()
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()

    # 2️⃣ Configurações de SFT (substitui TrainingArguments)
    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR / split_name),
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        fp16=True,
        optim="adamw_torch",
        learning_rate=5e-5,
        logging_strategy="steps",
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        dataloader_pin_memory=True,
        dataloader_num_workers=4,
        max_seq_length=512,
    )

    # 3️⃣ Data collator (padrão para language modeling)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8,
    )

    # 4️⃣ Instancia o SFTTrainer com peft_config
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=ds_train,



        eval_dataset=ds_test,
        data_collator=data_collator,
        peft_config=peft_config,
    )

    # 5️⃣ Treina e salva o adapter
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / split_name))
    print(f"✔️ Adapter salvo em {OUTPUT_DIR / split_name}")



=== Split: len ===


c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\peft\tuners\lora\layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


trainable params: 2,949,120 || all params: 776,979,200 || trainable%: 0.3796


Truncating eval dataset: 100%|██████████| 1054/1054 [00:00<?, ? examples/s]
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,1.913800,1.896518


In [ ]:
#  




=== Split: len ===
trainable params: 2,949,120 || all params: 776,979,200 || trainable%: 0.3796


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

In [ ]:
import bitsandbytes as bnb
hasattr(bnb.nn, "Linear4bit")

AttributeError: module 'bitsandbytes' has no attribute 'nn'